In [1]:
## Proyecto de PAMD
## Juan Esteban Roa y Sara Camila Aponte

In [2]:
from pyspark.sql import SparkSession

# Inicialización de Spark con las dependencias necesarias para la conexión con MinIO
spark = (
    SparkSession.builder
    .appName("PAMD")
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,"
        "/home/jovyan/work/jars/aws-java-sdk-bundle-1.11.1026.jar"
    )
    .getOrCreate()
)

print("Spark iniciado correctamente")
print(f"Versión de Spark: {spark.version}")

Spark iniciado correctamente
Versión de Spark: 3.5.0


In [3]:
# Verificación de los JAR cargados en la sesión de Spark
print(spark.sparkContext._jsc.sc().listJars())

Vector(spark://f3cb80160694:36655/jars/aws-java-sdk-bundle-1.11.1026.jar, spark://f3cb80160694:36655/jars/hadoop-aws-3.3.4.jar)


In [4]:
# Configuración de Hadoop para acceder a MinIO mediante S3A
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

print("Configuración de MinIO aplicada correctamente.")

Configuración de MinIO aplicada correctamente.


In [5]:
# Ruta de la capa Raw almacenada en MinIO
path = "s3a://raw/secop/"

try:
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path),
        hadoop_conf
    )

    archivos = fs.listStatus(
        spark._jvm.org.apache.hadoop.fs.Path(path)
    )

    print("Conexión Spark → MinIO exitosa")
    print("Archivos encontrados:")

    for archivo in archivos:
        print(f" - {archivo.getPath().toString()}")

except Exception as e:
    print("Error al conectar con MinIO:")
    print(e)

Conexión Spark → MinIO exitosa
Archivos encontrados:
 - s3a://raw/secop/datos_secop.csv


In [6]:
# Ruta del archivo Raw almacenado en MinIO
path = "s3a://raw/secop/datos_secop.csv"

# Lectura del CSV mediante PySpark
datos_crudos = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(path)
)

print("Dataset Raw cargado correctamente.")

Dataset Raw cargado correctamente.


In [7]:
# Perfilamiento inicial del dataset
numero_filas = datos_crudos.count()
numero_columnas = len(datos_crudos.columns)

print(f"Filas del dataset: {numero_filas:,}")
print(f"Columnas del dataset: {numero_columnas}")

Filas del dataset: 1,777,673
Columnas del dataset: 85


In [ ]:
# Inspección del esquema inicial
datos_crudos.printSchema()

In [18]:
from pyspark.sql.functions import col, sum, when

# Conteo de valores nulos o cadenas vacías por variable
nulos = datos_crudos.select([
    sum(
        when(
            col(c).isNull() | (col(c) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in datos_crudos.columns
])

nulos_vertical = nulos.selectExpr(
    "stack({0}, {1}) as (Variable, Nulos)".format(
        len(nulos.columns),
        ", ".join([f"'{c}', `{c}`" for c in nulos.columns])
    )
)

nulos_vertical.orderBy(col("Nulos").desc()).show(85, truncate=False)

+----------------------------------------------------------------------------+-------+
|Variable                                                                    |Nulos  |
+----------------------------------------------------------------------------+-------+
|Fecha de notificación de prorrogación                                       |1606059|
|Fecha Fin Liquidacion                                                       |1583140|
|Fecha Inicio Liquidacion                                                    |1583129|
|Ultima Actualizacion                                                        |748849 |
|Fecha de Inicio del Contrato                                                |139618 |
|Fecha de Firma                                                              |125974 |
|Fecha de Fin del Contrato                                                   |16214  |
|Número de cuenta                                                            |22     |
|Nit Entidad                               

In [8]:
from pyspark.sql import functions as F

# Variables temporales principales del contrato
columnas_fecha = [
    "Fecha de Firma",
    "Fecha de Inicio del Contrato",
    "Fecha de Fin del Contrato"
]

# Conversión de texto a tipo DATE
for columna in columnas_fecha:
    datos_crudos = datos_crudos.withColumn(
        columna,
        F.to_date(F.col(columna), "MM/dd/yyyy")
    )

print("Fechas principales convertidas a tipo DATE.")

Fechas principales convertidas a tipo DATE.


In [9]:
# Variables monetarias que requieren conversión numérica
columnas_monetarias = [
    "Valor del Contrato",
    "Valor de pago adelantado",
    "Valor Facturado",
    "Valor Pendiente de Pago",
    "Valor Pagado",
    "Valor Amortizado",
    "Valor Pendiente de Amortizacion",
    "Valor Pendiente de Ejecucion",
    "Saldo CDP",
    "Saldo Vigencia"
]

# Eliminación de símbolos y conversión a tipo DOUBLE
for columna in columnas_monetarias:
    datos_crudos = datos_crudos.withColumn(
        columna,
        F.regexp_replace(
            F.regexp_replace(F.col(columna), r"\$", ""),
            r"\.",
            ""
        ).cast("double")
    )

print("Valores monetarios convertidos a DOUBLE.")

Valores monetarios convertidos a DOUBLE.


In [10]:
# Conversión de días adicionados a tipo entero
datos_crudos = datos_crudos.withColumn(
    "Dias adicionados",
    F.col("Dias adicionados").cast("int")
)

print("Días adicionados convertidos a INTEGER.")

Días adicionados convertidos a INTEGER.


In [11]:
# Cálculo de la duración contractual en días
datos_crudos = datos_crudos.withColumn(
    "Duracion_Calculada_Dias",
    F.datediff(
        F.col("Fecha de Fin del Contrato"),
        F.col("Fecha de Inicio del Contrato")
    )
)

print("Duración contractual calculada correctamente.")

Duración contractual calculada correctamente.


In [12]:
# Identificación de registros incompletos según campos esenciales
registros_incompletos = datos_crudos.filter(
    F.col("ID Contrato").isNull() |
    F.col("Fecha de Firma").isNull() |
    F.col("Valor del Contrato").isNull()
)

cantidad_incompletos = registros_incompletos.count()
print(f"Registros incompletos identificados: {cantidad_incompletos:,}")

Registros incompletos identificados: 156,856


In [13]:
# Conservación de registros que contienen los campos esenciales
datos_bronce = datos_crudos.filter(
    F.col("ID Contrato").isNotNull() &
    F.col("Fecha de Firma").isNotNull() &
    F.col("Valor del Contrato").isNotNull()
)

cantidad_bronce = datos_bronce.count()
print(f"Registros válidos en Bronze: {cantidad_bronce:,}")

Registros válidos en Bronze: 1,620,817


In [14]:
# Identificación de registros con inconsistencias en las fechas contractuales
registros_fechas_corruptas = datos_bronce.filter(
    (F.col("Fecha de Inicio del Contrato") < F.col("Fecha de Firma")) |
    (F.col("Fecha de Fin del Contrato") < F.col("Fecha de Inicio del Contrato")) |
    (F.year(F.col("Fecha de Inicio del Contrato")) < 2000) |
    (F.year(F.col("Fecha de Fin del Contrato")) < 2000) |
    (F.year(F.col("Fecha de Fin del Contrato")) > 2100)
)

cantidad_fechas_corruptas = registros_fechas_corruptas.count()

print(
    f"Registros con fechas potencialmente corruptas: "
    f"{cantidad_fechas_corruptas:,}"
)

Registros con fechas potencialmente corruptas: 61,453


In [15]:
# Filtrado de inconsistencias temporales
datos_bronce_final = datos_bronce.filter(
    (F.col("Fecha de Inicio del Contrato") >= F.col("Fecha de Firma")) &
    (F.col("Fecha de Fin del Contrato") >= F.col("Fecha de Inicio del Contrato")) &
    (F.year(F.col("Fecha de Inicio del Contrato")) >= 2000) &
    (F.year(F.col("Fecha de Fin del Contrato")) >= 2000) &
    (F.year(F.col("Fecha de Fin del Contrato")) <= 2100)
)

cantidad_bronce_final = datos_bronce_final.count()
print(f"Registros en Bronze final: {cantidad_bronce_final:,}")

Registros en Bronze final: 1,535,493


In [16]:
# Ruta del Bronze final almacenado en Parquet sin particionar
ruta_bronce = "s3a://raw/procesado/contratos_bronze/"

# Almacenamiento del Bronze final en formato Parquet
datos_bronce_final.write \
    .mode("overwrite") \
    .parquet(ruta_bronce)

print("Bronze final almacenado correctamente en Parquet sin particionar.")
print(f"Ruta: {ruta_bronce}")

Bronze final almacenado correctamente en Parquet sin particionar.
Ruta: s3a://raw/procesado/contratos_bronze/


In [17]:
# Almacenamiento del Bronze en formato Parquet particionado
ruta_bronce_particionado = "s3a://raw/procesado/contratos_bronze_particionado/"

datos_bronce_final.write \
    .mode("overwrite") \
    .partitionBy("Tipo de Contrato") \
    .parquet(ruta_bronce_particionado)

print("Parquet particionado creado correctamente.")
print("Variable de particionamiento: Tipo de Contrato")
print(f"Ruta: {ruta_bronce_particionado}")

Parquet particionado creado correctamente.
Variable de particionamiento: Tipo de Contrato
Ruta: s3a://raw/procesado/contratos_bronze_particionado/


In [18]:
# Almacenamiento del bronze final en CSV

ruta_csv_bronce = "s3a://raw/procesado/contratos_bronze_csv/"

datos_bronce_final.write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv(ruta_csv_bronce)

print("CSV Bronze creado correctamente.")
print(f"Registros: {datos_bronce_final.count():,}")
print(f"Ruta: {ruta_csv_bronce}")

CSV Bronze creado correctamente.
Registros: 1,535,493
Ruta: s3a://raw/procesado/contratos_bronze_csv/


In [19]:
# Carga del Bronze final almacenado en formato CSV
datos_bronce_csv = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(ruta_csv_bronce)
)

print("Bronze CSV cargado correctamente.")

Bronze CSV cargado correctamente.


In [20]:
# Carga de las tres versiones físicas del Bronze final

datos_bronce_parquet = spark.read.parquet(ruta_bronce)

datos_bronce_particionado = (
    spark.read
    .parquet(ruta_bronce_particionado)
)

print("Tres versiones del Bronze final cargadas correctamente.")

Tres versiones del Bronze final cargadas correctamente.


In [21]:
# Consulta analítica sobre el Bronze final en formato CSV

consulta_csv = (
    datos_bronce_csv
    .filter(
        (F.col("Tipo de Contrato") == "Prestación de servicios") &
        (F.col("Departamento") == "Distrito Capital de Bogotá") &
        (F.col("Fecha de Firma") >= "2025-01-01") &
        (F.col("Fecha de Firma") <= "2026-08-24")
    )
    .agg(
        F.countDistinct("ID Contrato").alias("Cantidad_Contratos"),
        F.sum("Valor del Contrato").alias("Valor_Total_Contratado"),
        F.sum("Valor Pagado").alias("Valor_Total_Pagado"),
        F.avg("Valor del Contrato").alias("Valor_Promedio_Contrato"),
        F.avg("Duracion_Calculada_Dias").alias("Duracion_Promedio_Dias"),
        (
            F.sum("Valor Pagado") /
            F.sum("Valor del Contrato") * 100
        ).alias("Porcentaje_Total_Pagado")
    )
)

consulta_csv.show(truncate=False)

+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+
|Cantidad_Contratos|Valor_Total_Contratado|Valor_Total_Pagado|Valor_Promedio_Contrato|Duracion_Promedio_Dias|Porcentaje_Total_Pagado|
+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+
|130524            |1.0000070857514E13    |4.356773842856E12 |7.661480538072692E7    |236.75199963225154    |43.56742972058386      |
+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+



In [22]:
# Consulta analítica sobre el Bronze final en Parquet sin particionar

consulta_parquet_sin = (
    datos_bronce_parquet
    .filter(
        (F.col("Tipo de Contrato") == "Prestación de servicios") &
        (F.col("Departamento") == "Distrito Capital de Bogotá") &
        (F.col("Fecha de Firma") >= "2025-01-01") &
        (F.col("Fecha de Firma") <= "2026-08-24")
    )
    .agg(
        F.countDistinct("ID Contrato").alias("Cantidad_Contratos"),
        F.sum("Valor del Contrato").alias("Valor_Total_Contratado"),
        F.sum("Valor Pagado").alias("Valor_Total_Pagado"),
        F.avg("Valor del Contrato").alias("Valor_Promedio_Contrato"),
        F.avg("Duracion_Calculada_Dias").alias("Duracion_Promedio_Dias"),
        (
            F.sum("Valor Pagado") /
            F.sum("Valor del Contrato") * 100
        ).alias("Porcentaje_Total_Pagado")
    )
)

consulta_parquet_sin.show(truncate=False)

+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+
|Cantidad_Contratos|Valor_Total_Contratado|Valor_Total_Pagado|Valor_Promedio_Contrato|Duracion_Promedio_Dias|Porcentaje_Total_Pagado|
+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+
|130524            |1.0000070857514E13    |4.356773842856E12 |7.661480538072692E7    |236.75199963225154    |43.56742972058386      |
+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+



In [23]:
# Consulta analítica sobre el Bronze final en Parquet particionado

consulta_parquet_particionado = (
    datos_bronce_particionado
    .filter(
        (F.col("Tipo de Contrato") == "Prestación de servicios") &
        (F.col("Departamento") == "Distrito Capital de Bogotá") &
        (F.col("Fecha de Firma") >= "2025-01-01") &
        (F.col("Fecha de Firma") <= "2026-08-24")
    )
    .agg(
        F.countDistinct("ID Contrato").alias("Cantidad_Contratos"),
        F.sum("Valor del Contrato").alias("Valor_Total_Contratado"),
        F.sum("Valor Pagado").alias("Valor_Total_Pagado"),
        F.avg("Valor del Contrato").alias("Valor_Promedio_Contrato"),
        F.avg("Duracion_Calculada_Dias").alias("Duracion_Promedio_Dias"),
        (
            F.sum("Valor Pagado") /
            F.sum("Valor del Contrato") * 100
        ).alias("Porcentaje_Total_Pagado")
    )
)

consulta_parquet_particionado.show(truncate=False)

+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+
|Cantidad_Contratos|Valor_Total_Contratado|Valor_Total_Pagado|Valor_Promedio_Contrato|Duracion_Promedio_Dias|Porcentaje_Total_Pagado|
+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+
|130524            |1.0000070857514E13    |4.356773842856E12 |7.661480538072692E7    |236.75199963225154    |43.56742972058386      |
+------------------+----------------------+------------------+-----------------------+----------------------+-----------------------+



In [24]:
# Almacenamiento del resultado de la consulta analítica 1
ruta_resultado_consulta_1 = (
    "s3a://raw/procesado/consulta_prestacion_bogota/"
)

consulta_parquet_particionado.write \
    .mode("overwrite") \
    .parquet(ruta_resultado_consulta_1)

print("Resultado de la consulta 1 guardado correctamente.")
print(f"Ruta: {ruta_resultado_consulta_1}")

Resultado de la consulta 1 guardado correctamente.
Ruta: s3a://raw/procesado/consulta_prestacion_bogota/


In [25]:
import time

# Función para medir el tiempo de ejecución de una consulta analítica
def medir_tiempo(df, nombre):
    inicio = time.perf_counter()

    (
        df
        .filter(
            (F.col("Tipo de Contrato") == "Prestación de servicios") &
            (F.col("Departamento") == "Distrito Capital de Bogotá") &
            (F.col("Fecha de Firma") >= "2025-01-01") &
            (F.col("Fecha de Firma") <= "2026-08-24")
        )
        .agg(
            F.countDistinct("ID Contrato").alias("Cantidad_Contratos"),
            F.sum("Valor del Contrato").alias("Valor_Total_Contratado"),
            F.sum("Valor Pagado").alias("Valor_Total_Pagado"),
            F.avg("Valor del Contrato").alias("Valor_Promedio_Contrato"),
            F.avg("Duracion_Calculada_Dias").alias("Duracion_Promedio_Dias"),
            (
                F.sum("Valor Pagado") /
                F.sum("Valor del Contrato") * 100
            ).alias("Porcentaje_Total_Pagado")
        )
        .collect()
    )

    tiempo = time.perf_counter() - inicio

    print(f"{nombre}: {tiempo:.4f} segundos")

    return tiempo

In [28]:
# Medición del tiempo de ejecución sobre las tres versiones del dataset

tiempo_csv = medir_tiempo(
    datos_bronce_csv,
    "CSV Bronze"
)

tiempo_parquet_sin = medir_tiempo(
    datos_bronce_parquet,
    "Parquet sin particionar"
)

tiempo_parquet_particionado = medir_tiempo(
    datos_bronce_particionado,
    "Parquet particionado"
)

CSV Bronze: 53.8489 segundos
Parquet sin particionar: 4.9318 segundos
Parquet particionado: 3.9399 segundos


In [29]:
# Análisis de contratos según tipo de contrato y destino del gasto

consulta_destino = (
    datos_bronce_particionado
    .groupBy(
        "Tipo de Contrato",
        "Destino Gasto"
    )
    .agg(
        F.countDistinct("ID Contrato").alias("Cantidad_Contratos"),
        F.sum("Valor del Contrato").alias("Valor_Total_Contratado")
    )
    .orderBy(
        F.desc("Cantidad_Contratos")
    )
)

consulta_destino.show(truncate=False)

+--------------------------+--------------+------------------+----------------------+
|Tipo de Contrato          |Destino Gasto |Cantidad_Contratos|Valor_Total_Contratado|
+--------------------------+--------------+------------------+----------------------+
|Prestación de servicios   |Inversión     |877451            |1.593894685332196E15  |
|Prestación de servicios   |Funcionamiento|459246            |1.783724056692559E15  |
|Decreto 092 de 2017       |Funcionamiento|37791             |2.3276811566216E13    |
|Otro                      |Inversión     |36748             |6.396327283239086E15  |
|Suministros               |Funcionamiento|20155             |6.690096315812E12     |
|Otro                      |Funcionamiento|13810             |1.1542687203414E13    |
|Compraventa               |Funcionamiento|13535             |5.201768888287E12     |
|Decreto 092 de 2017       |Inversión     |13373             |7.691884335935E12     |
|Compraventa               |Inversión     |12142      

In [30]:
# Formateo del resultado para presentación
consulta_destino_formateada = consulta_destino.select(
    "Tipo de Contrato",
    "Destino Gasto",
    F.format_number(
        "Cantidad_Contratos", 0
    ).alias("Cantidad_Contratos"),
    F.format_number(
        "Valor_Total_Contratado", 0
    ).alias("Valor_Total_Contratado")
)

# Visualización de los 15 principales resultados
consulta_destino_formateada.limit(15).show(truncate=False)

+--------------------------+--------------+------------------+----------------------+
|Tipo de Contrato          |Destino Gasto |Cantidad_Contratos|Valor_Total_Contratado|
+--------------------------+--------------+------------------+----------------------+
|Prestación de servicios   |Inversión     |877,451           |1,593,894,685,332,196 |
|Prestación de servicios   |Funcionamiento|459,246           |1,783,724,056,692,559 |
|Decreto 092 de 2017       |Funcionamiento|37,791            |23,276,811,566,216    |
|Otro                      |Inversión     |36,748            |6,396,327,283,239,086 |
|Suministros               |Funcionamiento|20,155            |6,690,096,315,812     |
|Otro                      |Funcionamiento|13,810            |11,542,687,203,414    |
|Compraventa               |Funcionamiento|13,535            |5,201,768,888,287     |
|Decreto 092 de 2017       |Inversión     |13,373            |7,691,884,335,935     |
|Compraventa               |Inversión     |12,142     

In [31]:
# Almacenamiento del resultado de la consulta analítica 2
ruta_resultado_2 = (
    "s3a://raw/procesado/consulta_tipo_contrato_destino_gasto/"
)

consulta_destino_formateada.write \
    .mode("overwrite") \
    .parquet(ruta_resultado_2)

print("Consulta 2 guardada correctamente en MinIO.")
print(f"Ruta: {ruta_resultado_2}")

Consulta 2 guardada correctamente en MinIO.
Ruta: s3a://raw/procesado/consulta_tipo_contrato_destino_gasto/


In [32]:
# Comparación del tamaño físico

ruta_csv_bronce = "s3a://raw/procesado/contratos_bronze_csv/"
ruta_parquet_sin = "s3a://raw/procesado/contratos_bronze/"
ruta_parquet_particionado = "s3a://raw/procesado/contratos_bronze_particionado/"


# Función para calcular el tamaño físico almacenado
def obtener_tamano_recursivo(path):
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path),
        hadoop_conf
    )

    def recorrer(ruta):
        total = 0

        archivos = fs.listStatus(
            spark._jvm.org.apache.hadoop.fs.Path(ruta)
        )

        for archivo in archivos:
            if archivo.isFile():
                total += archivo.getLen()
            elif archivo.isDirectory():
                total += recorrer(archivo.getPath().toString())

        return total

    return recorrer(path)


# Cálculo de tamaños
tam_csv_bronce = obtener_tamano_recursivo(ruta_csv_bronce)
tam_parquet_sin = obtener_tamano_recursivo(ruta_parquet_sin)
tam_parquet_part = obtener_tamano_recursivo(ruta_parquet_particionado)


# Resultados
print("💾 TAMAÑO FÍSICO ALMACENADO")
print(f"CSV Bronze:                    {tam_csv_bronce / (1024**3):.2f} GiB")
print(f"Parquet sin particionar:       {tam_parquet_sin / (1024**3):.2f} GiB")
print(f"Parquet particionado:          {tam_parquet_part / (1024**3):.2f} GiB")

💾 TAMAÑO FÍSICO ALMACENADO
CSV Bronze:                    2.36 GiB
Parquet sin particionar:       0.64 GiB
Parquet particionado:          0.62 GiB
